# 📖 Running Evals with LangSmith

---

## 🎯 Learning Objectives

1. Run eval pipeline with LangSmith
2. Interpret results
3. Improve agents based on eval results

---

## ⏱️ Time Estimate
**~30 minutes**

In [ ]:
!pip install -q langsmith langchain langchain-openai
import os

# Would need actual API keys for full functionality
print("✅ Dependencies installed")

## 📊 The Eval Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│              EVAL PIPELINE                                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Dataset ──→ Agent ──→ Output ──→ Evaluator ──→ Results  │
│     │                    │                    │            │
│     │                    │                    │            │
│  Test cases         Predictions       Pass/Fail + Score   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 💻 Manual Eval Loop

In [ ]:
# Simulated eval loop
from openai import OpenAI
client = OpenAI()

# Simple agent
def simple_agent(query: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}]
    )
    return response.choices[0].message.content

# Simple evaluator
def simple_eval(input_query: str, reference: str, prediction: str) -> dict:
    correct = reference.lower() in prediction.lower()
    return {
        "passed": correct,
        "score": 100 if correct else 0,
        "feedback": "Correct" if correct else "Missing expected answer"
    }

# Dataset
test_cases = [
    ("What is 2+2?", "4"),
    ("Capital of France?", "Paris"),
    ("Python is what type of language?", "programming"),
]

# Run evals
print("📊 Running Evals...")
print("=" * 60)

all_results = []
for query, expected in test_cases:
    # Run agent
    prediction = simple_agent(query)
    
    # Evaluate
    result = simple_eval(query, expected, prediction)
    all_results.append(result)
    
    # Print
    status = "✅" if result["passed"] else "❌"
    print(f"{status} Query: {query}")
    print(f"   Expected: {expected}")
    print(f"   Got: {prediction[:50]}...")
    print()

# Summary
passed = sum(1 for r in all_results if r["passed"])
total = len(all_results)
print(f"📈 Summary: {passed}/{total} passed ({100*passed/total:.0f}%)")

## 🔧 LangSmith Evaluate API

In [ ]:
# LangSmith evaluate() function
# from langsmith import evaluate
# from langchain_openai import ChatOpenAI

# def predict_conversation(messages):
#     llm = ChatOpenAI(model="gpt-4o-mini")
#     return llm.invoke(messages).content

# # Define evaluator
# def my_evaluator(run, example):
#     return {
#         "key": "accuracy",
#         "score": 1.0 if "correct" in run.outputs["output"].lower() else 0.0
#     }

# # Run
# evaluate(
#     predict_conversation,
#     data="my-dataset",
#     evaluators=[my_evaluator],
#     experiment_prefix="my-agent-eval"
# )

print("""
# LangSmith CLI Command
langsmith evaluate \
  --agent my_agent.py \
  --dataset "my-test-dataset" \
  --evaluators accuracy \
  --project "agent-eval-results"
""")

## 📊 Understanding Results

In [ ]:
print("""
┌─────────────────────────────────────────────────────────┐
│         INTERPRETING EVAL RESULTS                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  SCORES:                                               │
│  • 90-100%: Excellent, production ready               │
│  • 70-90%:  Good, minor improvements needed          │
│  • 50-70%:  Needs work, significant issues            │
│  • <50%:    Major problems, don't deploy              │
│                                                         │
│  WHAT TO CHECK:                                        │
│  • Which test cases failed?                           │
│  • Are failures consistent?                          │
│  • Are failures in specific categories?               │
│  • Is there a pattern?                                │
│                                                         │
│  IMPROVING:                                            │
│  • Add more test cases for failing categories         │
│  • Improve tool descriptions                         │
│  • Add guardrails for safety                         │
│  • Tune prompts based on failures                    │
│                                                         │
└─────────────────────────────────────────────────────────┘
""")

## ✅ Summary

You learned:
1. **Eval pipeline** - Dataset → Agent → Evaluator → Results
2. **Running evals** - Both programmatic and CLI
3. **Interpreting results** - Scores, patterns, improvements

## 🔗 Next
**[06_error_propagation_examples.ipynb](06_error_propagation_examples.ipynb)** - See errors in action!